In [ ]:
import requestsimport reimport openpyxlimport pandas as pdfrom pathlib import Path

In [ ]:
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; AtlasFinanceBot/1.0)"}API_URL = "https://data.gov.ma/data/api/3/action/package_search"params = {"fq": "organization:bank-al-maghrib", "q": "ventilation credit bancaire", "rows": 20}resp = requests.get(API_URL, params=params, headers=HEADERS, timeout=30)data = resp.json()data["result"]["count"]

In [ ]:
packages = data["result"]["results"][p["title"] for p in packages]

In [ ]:
target_pkg = Nonefor p in packages:    if "ventilation" in p["title"].lower():        target_pkg = p        breaktarget_pkg["title"]

In [ ]:
resource = Nonefor r in target_pkg["resources"]:    if r.get("format", "").lower() in ("xlsx", "xls"):        resource = r        breakresource["url"]

In [ ]:
raw_folder = Path("../data/raw/bam")raw_folder.mkdir(parents=True, exist_ok=True)filename = resource["url"].split("/")[-1]filepath = raw_folder / filenamer = requests.get(resource["url"], headers=HEADERS, timeout=60)with open(filepath, "wb") as f:    f.write(r.content)filepath

In [ ]:
wb = openpyxl.load_workbook(filepath, data_only=True)ws = wb[wb.sheetnames[0]]ws.dimensions

In [ ]:
dates = []col = 2header_row = 3while ws.cell(row=header_row, column=col).value is not None:    dates.append(ws.cell(row=header_row, column=col).value)    col += 1len(dates), dates[0], dates[-1]

In [ ]:
records = []current_categorie = Nonecurrent_secteur = Nonerow = header_row + 1while True:    label_cell = ws.cell(row=row, column=1)    if label_cell.value is None:        break    label = str(label_cell.value).strip()    if label.startswith("(1)") or label.startswith("(2)"):        break    indent = label_cell.alignment.indent if label_cell.alignment else 0    if indent == 0:        current_categorie = label        current_secteur = None        niveau = "categorie"        secteur_val = None    elif indent == 2:        current_secteur = label        niveau = "secteur"        secteur_val = label    else:        niveau = "sous_detail"        secteur_val = current_secteur    for i in range(len(dates)):        value = ws.cell(row=row, column=2 + i).value        records.append({            "date": dates[i],            "categorie_objet": current_categorie,            "secteur_institutionnel": secteur_val,            "libelle_detail": label if niveau == "sous_detail" else None,            "niveau": niveau,            "encours_mdh": value        })    row += 1

In [ ]:
df_bam = pd.DataFrame(records)df_bam["date"] = pd.to_datetime(df_bam["date"])df_bam["encours_mdh"] = pd.to_numeric(df_bam["encours_mdh"], errors="coerce")df_bam.shape

In [ ]:
df_bam.head(20)

In [ ]:
df_bam.to_csv("../data/raw/bam_extracted.csv", index=False)